# **Activate GPU**

In [ ]:
import tensorflow as tf
print("TensorFlow version:", tf.__version__)
print("GPU available:", tf.config.list_physical_devices('GPU'))


TensorFlow version: 2.19.0
GPU available: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


# **Add Important Libraries**

In [ ]:
import os
import tensorflow as tf
from tensorflow import keras
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import ConfusionMatrixDisplay

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.preprocessing import image
from tensorflow.keras.models import Sequential,Model
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense,Dropout,Rescaling,RandomFlip,RandomRotation,RandomZoom,BatchNormalization,GlobalAveragePooling2D,RandomContrast,RandomBrightness,Activation


from tensorflow.keras.initializers import HeNormal
from tensorflow.keras import layers
from tensorflow.data import AUTOTUNE
from tensorflow.keras.optimizers import Adam
from keras.callbacks import EarlyStopping

from tensorflow.keras.models import load_model

# **Add Google Drive**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


# **Add Kaggle API**

In [ ]:
from google.colab import files
files.upload()



# **Add Dataset From Kaggle**

In [ ]:
!mkdir -p ~/.kaggle


!mv kaggle.json ~/.kaggle/


!chmod 600 ~/.kaggle/kaggle.json

!kaggle datasets download -d rizwan123456789/potato-disease-leaf-datasetpld

!unzip -q potato-disease-leaf-datasetpld.zip -d ./data

!rm potato-disease-leaf-datasetpld.zip

!ls ./data

Dataset URL: https://www.kaggle.com/datasets/rizwan123456789/potato-disease-leaf-datasetpld
License(s): DbCL-1.0
  0% 0.00/37.4M [00:00<?, ?B/s]
100% 37.4M/37.4M [00:00<00:00, 1.19GB/s]
PLD_3_Classes_256


# **Set Data Directory**

In [ ]:
data_dir = "/content/data/PLD_3_Classes_256"

train_dir = os.path.join(data_dir, "Training")
val_dir   = os.path.join(data_dir, "Validation")
test_dir  = os.path.join(data_dir, "Testing")

# **Split The Data into Train and Test Set**

In [ ]:
train_ds=keras.utils.image_dataset_from_directory(
    train_dir,
    image_size=(224,224),
    batch_size=64,
    shuffle=True,
    color_mode="rgb",
    label_mode="categorical",
    labels="inferred"
)
val_ds=keras.utils.image_dataset_from_directory(
    val_dir,
    image_size=(224,224),
    batch_size=64,
    shuffle=False,
    color_mode="rgb",
    label_mode="categorical",
    labels="inferred"
)
test_ds=keras.utils.image_dataset_from_directory(
    test_dir,
    image_size=(224,224),
    batch_size=64,
    shuffle=False,
    color_mode="rgb",
    label_mode="categorical",
    labels="inferred"
)

Found 3251 files belonging to 3 classes.
Found 416 files belonging to 3 classes.
Found 405 files belonging to 3 classes.


## **Find The Class Name**

In [ ]:
class_names=train_ds.class_names
print(class_names)

['Early_Blight', 'Healthy', 'Late_Blight']


# **Perform Cache and Autotune for Fast Processing**

In [ ]:
train_ds=train_ds.cache().shuffle(2500).prefetch(buffer_size=AUTOTUNE)
test_ds=test_ds.cache().prefetch(buffer_size=AUTOTUNE)
val_ds=val_ds.cache().prefetch(buffer_size=AUTOTUNE)

# **Add Pretrained Model DenseNet121**

In [ ]:
from tensorflow.keras.applications import DenseNet121
base_model_densenet=DenseNet121(include_top=False,input_shape=(224,224,3),weights='imagenet')


29084464/29084464 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step


# **Perform Data Augmentation to Reduce Overfitting**

In [ ]:
data_aug_den=Sequential([
    RandomFlip("horizontal"),
    RandomRotation(0.2),
    RandomZoom(0.2)

])

# **Freeze Upper LAyer and Add Custom Dense Layer**

In [ ]:
#Freeze all layers
for layer in base_model_densenet.layers:
  layer.trainable = False


#unfreeze last 150 layers
for layer in base_model_densenet.layers[-150:]:
  layer.trainable=True


inp=layers.Input(shape=(224,224,3))

x=data_aug_den(inp)
x = base_model_densenet(x, training=True)

#add own fully connected layers

x=GlobalAveragePooling2D()(x)
x=Dense(256,activation="relu",kernel_initializer='he_normal')(x)
x=Dropout(0.4)(x)
x=Dense(256,activation="relu",kernel_initializer='he_normal')(x)
x=Dropout(0.4)(x)
output = Dense(3, activation="softmax")(x)


model_den=Model(inp,output)

In [ ]:
model_den.summary()

Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_3 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ sequential (Sequential)         │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ densenet121 (Functional)        │ (None, 7, 7, 1024)     │     7,037,504 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_1      │ (None, 1024)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 256)            │       262,400 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 256)            │        65,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 3)              │           771 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 7,366,467 (28.10 MB)

 Trainable params: 3,691,907 (14.08 MB)

 Non-trainable params: 3,674,560 (14.02 MB)

# **Compile The Model Using Adam Optimizer And Add Callback To Reduce Overfitting**

In [ ]:
model_den.compile(optimizer=Adam(learning_rate=1e-6),loss='categorical_crossentropy',metrics=['accuracy'])


early_stopping_den = EarlyStopping(monitor='val_loss', patience=10,verbose=1, restore_best_weights=True)

# **Train Densenet Model**

In [ ]:
history_den=model_den.fit(train_ds,epochs=80,validation_data=val_ds,verbose=1,callbacks=[early_stopping_den])

Epoch 1/80
51/51 ━━━━━━━━━━━━━━━━━━━━ 77s 596ms/step - accuracy: 0.2914 - loss: 1.9644 - val_accuracy: 0.2452 - val_loss: 1.9693
Epoch 2/80
51/51 ━━━━━━━━━━━━━━━━━━━━ 20s 403ms/step - accuracy: 0.3216 - loss: 1.8213 - val_accuracy: 0.2332 - val_loss: 1.5314
Epoch 3/80
51/51 ━━━━━━━━━━━━━━━━━━━━ 21s 406ms/step - accuracy: 0.3384 - loss: 1.7630 - val_accuracy: 0.2764 - val_loss: 1.2810
Epoch 4/80
51/51 ━━━━━━━━━━━━━━━━━━━━ 21s 410ms/step - accuracy: 0.3360 - loss: 1.6952 - val_accuracy: 0.3510 - val_loss: 1.1431
Epoch 5/80
51/51 ━━━━━━━━━━━━━━━━━━━━ 21s 413ms/step - accuracy: 0.3565 - loss: 1.5934 - val_accuracy: 0.4351 - val_loss: 1.0582
Epoch 6/80
51/51 ━━━━━━━━━━━━━━━━━━━━ 21s 414ms/step - accuracy: 0.3727 - loss: 1.5587 - val_accuracy: 0.5000 - val_loss: 1.0023
Epoch 7/80
51/51 ━━━━━━━━━━━━━━━━━━━━ 21s 417ms/step - accuracy: 0.3604 - loss: 1.5087 - val_accuracy: 0.5312 - val_loss: 0.9607
Epoch 8/80
51/51 ━━━━━━━━━━━━━━━━━━━━ 21s 417ms/step - accuracy: 0.4025 - loss: 1.4343 - val_accu

# **Save Our Densenet Model in Local Environment**

In [ ]:
model_den.save("potato_densenet_full_model.h5")


converter = tf.lite.TFLiteConverter.from_keras_model(model_den)
tflite_model_vgg = converter.convert()

with open("potato_densenet_full_model.tflite", "wb") as f:
    f.write(tflite_model_vgg)


Saved artifact at '/tmp/tmpu460vkve'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 224, 224, 3), dtype=tf.float32, name='keras_tensor_440')
Output Type:
  TensorSpec(shape=(None, 3), dtype=tf.float32, name=None)
Captures:
  132368208875536: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132368208878032: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132368208876112: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132368208877072: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132368208877456: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132368208878224: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132368208876880: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132368208877264: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132368208877648: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132368208879568: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1323682088

# **Load The Densenet Model**

In [ ]:
model_den= load_model(
    "/content/drive/MyDrive/cnn_full_model_densenet.h5"
)

# **Check The Accuracy And Loss**

In [ ]:
loss_den,acc_den=model_den.evaluate(test_ds)
print("Accuracy of DenseNet121: ",acc_den)
print("Loss of DenseNet121: ",loss_den)

7/7 ━━━━━━━━━━━━━━━━━━━━ 5s 731ms/step - accuracy: 0.8674 - loss: 0.3485
Accuracy of DenseNet121:  0.8172839283943176
Loss of DenseNet121:  0.4228265881538391


# **Evaluate Matrics**

In [ ]:
y_true = np.concatenate([y for x, y in test_ds], axis=0)
y_true = np.argmax(y_true, axis=1)

y_pred = np.argmax(model_den.predict(test_ds), axis=1)


print(" Confusion Matrix:")
print(confusion_matrix(y_true, y_pred))

print("\n Classification Report:")
print(classification_report(
    y_true, y_pred,
    target_names=['Early Blight', 'Healthy', 'Late Blight']
))


7/7 ━━━━━━━━━━━━━━━━━━━━ 9s 816ms/step
 Confusion Matrix:
[[155   1   6]
 [ 18  71  13]
 [ 27   9 105]]

 Classification Report:
              precision    recall  f1-score   support

Early Blight       0.78      0.96      0.86       162
     Healthy       0.88      0.70      0.78       102
 Late Blight       0.85      0.74      0.79       141

    accuracy                           0.82       405
   macro avg       0.83      0.80      0.81       405
weighted avg       0.83      0.82      0.81       405

